# Очистка и анализ данных такси

В этом ноутбуке мы выполним полный цикл обработки данных:
- Загрузка и предварительный анализ
- Очистка данных
- Объединение таблиц
- Трансформации и агрегации
- Сохранение результатов

## 1. Импорт библиотек и загрузка данных

In [214]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [215]:
# Загружаем таблицы
trips = pd.read_csv('data/trips.csv')
trip_details = pd.read_csv('data/trip_details.csv')

print("Таблицы загружены:")
print(f"  trips: {trips.shape}")
print(f"  trip_details: {trip_details.shape}")

Таблицы загружены:
  trips: (51, 5)
  trip_details: (100, 6)


## 2. Анализ данных

In [216]:
print("=== TRIPS ===")
display(trips.head())

print("=== TRIPS DTYPES ===")
print(trips.dtypes)

=== TRIPS ===


,trip_id,date,client_id,status,total_price
0,1,2024-01-01,103.00,cancelled,40
1,2,2024-01-02,104.00,cancelled,-15
2,3,2024-01-03,102.00,completed,30
3,4,2024-01-04,104.00,cancelled,30
4,5,2024-01-05,104.00,completed,10


=== TRIPS DTYPES ===
trip_id          int64
date               str
client_id      float64
status             str
total_price        str
dtype: object


In [217]:
print("=== TRIP DETAILS ===")
display(trip_details.head())

print("=== TRIP DETAILS DTYPES ===")
print(trip_details.dtypes)

=== TRIP DETAILS ===


,detail_id,trip_id,service,category,quantity,price
0,1,32,luggage,transport,-1,20
1,2,39,luggage,service,3,20
2,3,49,ride,transport,3,10
3,4,52,ride,service,-1,15
4,5,32,luggage,extra,2,20


=== TRIP DETAILS DTYPES ===
detail_id    int64
trip_id      int64
service        str
category       str
quantity     int64
price          str
dtype: object


In [218]:
def analyze_missing(df, name):
    """Анализ пропущенных значений в таблице."""
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

    result = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': missing_pct
    })

    print(f"\n=== {name} ===")
    print(result[result['missing_count'] > 0])

# Проверка
analyze_missing(trips, 'TRIPS')
analyze_missing(trip_details, 'TRIP_DETAILS')


=== TRIPS ===
           missing_count  missing_pct
client_id              8        15.69

=== TRIP_DETAILS ===
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


In [219]:
print(f"Дубликаты в trips: {trips.duplicated(subset=['trip_id']).sum()}")
print(f"Дубликаты в trip_details: {trip_details.duplicated(subset=['detail_id']).sum()}")

Дубликаты в trips: 1
Дубликаты в trip_details: 0


In [220]:
print("=== Некорректные даты в trips ===")

invalid_dates = trips[
    ~trips['date'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
]

print(f"Количество некорректных дат: {len(invalid_dates)}")

if len(invalid_dates) > 0:
    print("Примеры:")
    display(invalid_dates[['trip_id', 'date']].head())

=== Некорректные даты в trips ===
Количество некорректных дат: 2
Примеры:


,trip_id,date
5,6,2024/01/07
10,11,not_a_date


In [221]:
# === Приведение типов ===
trips['total_price'] = pd.to_numeric(trips['total_price'], errors='coerce')
trip_details['price'] = pd.to_numeric(trip_details['price'], errors='coerce')
trip_details['quantity'] = pd.to_numeric(trip_details['quantity'], errors='coerce')

# === Отрицательные значения ===
print("\n=== Отрицательные значения ===")

print(f"Отрицательный quantity: {(trip_details['quantity'] < 0).sum()}")
print(f"Отрицательный price: {(trip_details['price'] < 0).sum()}")

# 👇 ВАЖНО: теперь работает без ошибки
print(f"Отрицательный total_price: {(trips['total_price'] < 0).sum()}")


=== Отрицательные значения ===
Отрицательный quantity: 27
Отрицательный price: 0
Отрицательный total_price: 11


In [222]:
trip_details['item_total'] = trip_details['quantity'] * trip_details['price']

trip_totals = trip_details.groupby('trip_id')['item_total'].sum().reset_index()
trip_totals = trip_totals.rename(columns={'item_total': 'calculated_total'})

trips_with_calc = trips.merge(trip_totals, on='trip_id', how='left')

In [223]:
trips_with_calc['total_price'] = pd.to_numeric(trips_with_calc['total_price'], errors='coerce')
trips_with_calc['calculated_total'] = pd.to_numeric(trips_with_calc['calculated_total'], errors='coerce')

trips_with_calc['difference'] = trips_with_calc['total_price'] - trips_with_calc['calculated_total']
trips_with_calc['diff_pct'] = (trips_with_calc['difference'] / trips_with_calc['calculated_total'] * 100).round(2)

In [224]:
mismatched = trips_with_calc[
    (trips_with_calc['difference'].abs() > 0.01) &
    (trips_with_calc['calculated_total'].notna())
]

print("=== Проверка total_price ===")
print(f"Всего поездок: {len(trips)}")
print(f"С расхождениями: {len(mismatched)}")

display(mismatched[['trip_id', 'total_price', 'calculated_total', 'difference', 'diff_pct']].head())

=== Проверка total_price ===
Всего поездок: 51
С расхождениями: 33


,trip_id,total_price,calculated_total,difference,diff_pct
0,1,40.00,30.00,10.00,33.33
1,2,-15.00,40.00,-55.00,-137.50
2,3,30.00,45.00,-15.00,-33.33
3,4,30.00,15.00,15.00,100.00
4,5,10.00,0.00,10.00,inf


В данном блоке выполняется проверка согласованности данных между основной таблицей поездок и детализацией услуг.

Для каждой поездки рассчитывается фактическая стоимость на основе состава услуг, после чего она сравнивается с заявленной стоимостью.

Отклонения между значениями позволяют выявить:

ошибки в расчетах
некорректные данные
возможные системные сбои при формировании отчетов

Данный этап является важной частью контроля качества данных перед дальнейшим анализом.

## 3. Очистка данных

In [225]:
# создаём копии данных
trips_clean = trips.copy()
trip_details_clean = trip_details.copy()

In [226]:
trips_clean['total_price'] = pd.to_numeric(trips_clean['total_price'], errors='coerce')
trip_details_clean['price'] = pd.to_numeric(trip_details_clean['price'], errors='coerce')
trip_details_clean['quantity'] = pd.to_numeric(trip_details_clean['quantity'], errors='coerce')

In [227]:
print("Дубликаты trips:", trips_clean.duplicated(subset=['trip_id']).sum())
trips_clean = trips_clean.drop_duplicates(subset=['trip_id'], keep='first')

print("Дубликаты trip_details:", trip_details_clean.duplicated(subset=['detail_id']).sum())
trip_details_clean = trip_details_clean.drop_duplicates(subset=['detail_id'], keep='first')

Дубликаты trips: 1
Дубликаты trip_details: 0


In [228]:
print(trips_clean.isnull().sum())
print(trip_details_clean.isnull().sum())

trip_id        0
date           0
client_id      7
status         0
total_price    4
dtype: int64
detail_id      0
trip_id        0
service        0
category       0
quantity       0
price         23
item_total    23
dtype: int64


In [229]:
trips_clean['status'] = trips_clean['status'].fillna('unknown')

trip_details_clean['price'] = trip_details_clean['price'].fillna(trip_details_clean['price'].median())
trip_details_clean['quantity'] = trip_details_clean['quantity'].fillna(1)

In [230]:
trips_clean = trips_clean[trips_clean['total_price'] > 0]

trip_details_clean = trip_details_clean[
    (trip_details_clean['quantity'] > 0) &
    (trip_details_clean['price'] > 0)
]

In [231]:
valid_trip_ids = set(trips_clean['trip_id'])

trip_details_clean = trip_details_clean[
    trip_details_clean['trip_id'].isin(valid_trip_ids)
]

In [232]:
trip_details_clean['item_total'] = trip_details_clean['quantity'] * trip_details_clean['price']

trip_totals_clean = trip_details_clean.groupby('trip_id')['item_total'].sum().reset_index()
trip_totals_clean = trip_totals_clean.rename(columns={'item_total': 'calculated_total'})

trips_clean = trips_clean.merge(trip_totals_clean, on='trip_id', how='left')

In [233]:
trips_clean['difference'] = trips_clean['total_price'] - trips_clean['calculated_total']

print("Средняя ошибка:", trips_clean['difference'].abs().mean())
print("Максимальная ошибка:", trips_clean['difference'].abs().max())

Средняя ошибка: 23.333333333333332
Максимальная ошибка: 105.0


После выполнения этапа очистки данных были выполнены следующие шаги:

удалены дубликаты по уникальным идентификаторам

обработаны пропущенные значения с использованием логически обоснованных стратегий

устранены логические аномалии (отрицательные значения)

выполнена каскадная очистка связанных таблиц

приведены числовые признаки к корректным типам

Данные были приведены к согласованному виду и готовы для дальнейшего анализа и построения агрегированных метрик.

## 4. Объединение данных

In [234]:
# trips + trip_details
merged = trip_details_clean.merge(
    trips_clean,
    on='trip_id',
    how='inner',
    suffixes=('_item', '_trip')
)

In [235]:
merged['date'] = pd.to_datetime(merged['date'], errors='coerce')

In [236]:
display(merged.head())

,detail_id,trip_id,service,category,quantity,price,item_total,date,client_id,status,total_price,calculated_total,difference
0,3,49,ride,transport,3,10.00,30.00,2024-02-18,100.00,unknown,40.00,145.00,-105.00
1,5,32,luggage,extra,2,20.00,40.00,2024-02-01,101.00,unknown,50.00,130.00,-80.00
2,6,4,waiting,transport,3,5.00,15.00,2024-01-04,104.00,cancelled,30.00,15.00,15.00
3,7,30,ride,transport,2,5.00,10.00,2024-01-30,102.00,unknown,10.00,10.00,0.00
4,8,37,luggage,transport,1,10.00,10.00,2024-02-06,NaN,cancelled,50.00,50.00,0.00


In [237]:
# считаем общую выручку сервиса по месяцам
merged['year_month'] = merged['date'].dt.to_period('M')
merged['item_revenue'] = merged['quantity'] * merged['price']

revenue_by_month = merged.groupby('year_month')['item_revenue'].sum().reset_index()

display(revenue_by_month)

,year_month,item_revenue
0,2024-01,585.00
1,2024-02,435.00


In [238]:
top_clients = merged.groupby('client_id').agg(
    total_trips=('trip_id', 'nunique')
).reset_index()

top_clients = top_clients.sort_values('total_trips', ascending=False).head(10)

display(top_clients)

,client_id,total_trips
2,102.00,7
1,101.00,5
0,100.00,3
3,103.00,3
4,104.00,3


In [239]:
# какие услуги чаще всего приводят к отменам
return_statuses = ['cancelled', 'refunded']

returns = merged[merged['status'].isin(return_statuses)]

returns_by_service = returns.groupby('service').agg(
    num_returns=('trip_id', 'nunique'),
    qty_returned=('quantity', 'sum')
).sort_values('num_returns', ascending=False)

display(returns_by_service)

,num_returns,qty_returned
service,,
ride,6,13
waiting,5,12
luggage,4,14


In [240]:
# ПРОВЕРКА ПЕРЕПРОДАЖ
service_sold = merged.groupby('service')['quantity'].sum().reset_index()

stock_check = service_sold.merge(
    trip_details_clean[['service']].drop_duplicates(),
    on='service',
    how='left'
)

In [241]:
# сортируем по дате
merged['date'] = pd.to_datetime(merged['date'], errors='coerce')

# находим первую поездку клиента
first_trip = merged.groupby('client_id')['date'].min().reset_index()
first_trip = first_trip.rename(columns={'date': 'first_trip_date'})

# добавляем в merged
merged = merged.merge(first_trip, on='client_id', how='left')

# считаем разницу
merged['days_from_first_trip'] = (
    merged['date'] - merged['first_trip_date']
).dt.days

# новый клиент = первая поездка
merged['is_new_customer'] = merged['days_from_first_trip'] == 0

In [242]:
final_df = merged.copy()
final_df.to_csv('data/final_taxi_dataset.csv', index=False)